<a href="https://colab.research.google.com/github/anhquan-truong/PM520/blob/main/HW/PM520_HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 3. Variational Inference

## 1. Evidence Lower Bound
$\newcommand{\bX}{\mathbf{X}}\newcommand{\by}{\mathbf{y}}\newcommand{\bI}{\mathbf{I}}$
Recall from Lab 8, our example of variational inference for a Bayesian linear regression model. Namely,
$$\begin{align*}
\by | \bX, \beta &\sim N(\bX\beta, \bI_n \sigma^2) \\
\beta &\sim N(0, \bI_p \sigma^2_b).
\end{align*}$$

We assumed a mean-field model that $Q$ factorizes as $$Q(\beta) = \prod_{j=1}^P Q_j(\beta_j).$$

### 1.1
Consulting the results in Lab 8 on parameter definitions for each $Q_j$, please derive the *evidence lower bound* or ELBO for this model.
From Lab 8, we have the definition of $Q_j$ as

$$Q_j^*(\beta_j) = N(\beta_j | \tilde{\mu}_j, \widetilde{\sigma}^2_j)$$

$\newcommand{\data}{\text{Data}}$
$\newcommand{\E}{\mathbb{E}}$
$\newcommand{\ELBO}{\text{ELBO}}$

The $\ELBO$ definition is:
$$\begin{align*}
\ELBO &:= -\E_Q[ \log Q(\beta | \data)] + \E_Q[\log \Pr(\data | \beta)] + \E_Q[\log \Pr(\beta)] \\
  &= \E_Q[\log \Pr(\data | \beta)] - \E_Q\left[ \log \frac{Q(\beta | \data)}{\Pr(\beta)}\right]\\
  &= \E_Q[\log \Pr(\data | \beta)] - D_{KL}(Q(\beta | \data) || \Pr(\beta)).
\end{align*}$$

To derive the $\ELBO$, we will find the expectation of the log-likelihood with respect to $Q_\beta$

$$\begin{align*}
 \E_Q[\log \Pr(y | X, \beta)] &= \E_Q\left[-\frac{1}{2\sigma^2} (y-X\beta)^T(y-X\beta)  \right] + O(1) \\
 &= \E_Q\left[-\frac{1}{2\sigma^2} (y^Ty - 2y^TX\beta + \beta^TX^TX\beta) \right] + O(1) \\
  &= -\frac{1}{2\sigma^2} (\E_Q[y^Ty] - \E_Q[2y^TX\beta] + \E_Q[\beta^TX^TX\beta)]  + O(1) \\
  &= -\frac{1}{2\sigma^2} \left[y^Ty - 2y^TX \tilde{\mu} + \tilde{\mu}^TX^TX\tilde{\mu} + tr(X^TXI_d\widetilde{\sigma}^2)\right]  + O(1) \\
  &= -\frac{1}{2\sigma^2} \left[y^Ty - 2y^TX \tilde{\mu} + \tilde{\mu}^TX^TX\tilde{\mu} +\sum_{j=1}^d||X_j||^2_2 \widetilde{\sigma}^2 \right]  + O(1)
\end{align*}$$

The KL-divergence is:

$$\begin{align*}
D_{KL}(Q(β|Data)||Pr(β)) &= \E_Q\left[ \log \frac {Q(β|Data)}{Pr(β)}\right]\\
&= \E_Q\left[ -\frac{(\beta - \tilde{\mu})^2}{2\widetilde{\sigma}^2} +\frac{(\beta - 0)^2)}{2\sigma^2} \right] + \frac{1}{2}\log(2\pi\sigma^2) - \frac{1}{2}\log(2\pi\widetilde\sigma^2)\\
&= -\frac{\E_Q[(\beta - \tilde{\mu})^2]}{2\widetilde{\sigma}^2} + \frac{\E_Q[\beta^2)]}{2\sigma^2}  + \frac{1}{2}\log\frac{\sigma^2}{\widetilde\sigma^2} \\
&= -\frac{1}{2} + \frac{\tilde\mu^2 + \widetilde\sigma^2}{2\sigma^2}  + \frac{1}{2}\log\frac{\sigma^2}{\widetilde\sigma^2}\\
\end{align*}$$

### 1.2
Consult lab 8 for the implementation of a CAVI algorithm for the model above, but rather than evaluate the mean squared error (MSE), evaluate the ELBO. The ELBO should *increase* with each iteration, otherwise there is likely a bug.

In [4]:
import jax
import jax.numpy as jnp
import jax.random as rdm

MAX_ITER = 100
TOL = 0.01

N = 5000
P = 250
sigma_sq = 0.8
sigma_sq_b = 0.1

seed = 0
key = rdm.PRNGKey(seed)
key, x_key, b_key, y_key = rdm.split(key, 4)

X = rdm.normal(x_key, shape=(N, P))
beta = jnp.sqrt(sigma_sq_b) * rdm.normal(b_key, shape=(P,))
y = X @ beta + jnp.sqrt(sigma_sq) * rdm.normal(y_key, shape=(N,))

post_means = jnp.zeros((P,))
post_vars = jnp.ones((P,)) * sigma_sq_b

def kl_norm(means, vars, prior_var):
  return 0.5 * (means ** 2 + vars) / prior_var + (jnp.log(prior_var) - jnp.log(vars))

def compute_elbo(y, X, means, vars, prior_var):
  kls = jnp.sum(kl_norm(means, vars, prior_var))
  pred = X @ means
  ess = pred @ y - jnp.sum((X * X) @ vars) - pred @ pred
  return ess - kls

post_means = jnp.zeros((P,))

elbo = compute_elbo(y, X, post_means, post_vars, sigma_sq_b)
for _iter in range(MAX_ITER):
  r = y - X @ post_means
  for j in range(P):
    Xj = X[:,j]
    rj = r + Xj * post_means[j]
    post_vars_j = jnp.reciprocal(jnp.sum(Xj * Xj) / sigma_sq + 1. / sigma_sq_b)
    post_means_j = (post_vars_j / sigma_sq) * Xj.T @ rj
    post_means = post_means.at[j].set(post_means_j)
    post_vars = post_vars.at[j].set(post_vars_j)
    r = rj - Xj * post_means_j
  elbo_new = compute_elbo(y, X, post_means, post_vars, sigma_sq_b)
  print(f"ELBO[{_iter}] = {elbo_new}")
  diff = jnp.fabs(elbo_new - elbo)
  if diff < TOL:
    break
  elbo = elbo_new

print(f"post mean = {post_means}")

KeyboardInterrupt: 

## 2. Bayesian Linear Regression Pt II
Here we assume a slightly different linear model, which is given by, $$\begin{align*}
\by | \bX, \beta &\sim N(\bX\beta, \bI_n \sigma^2) \\
\beta_j &\sim \text{Laplace}(0, b).
\end{align*}$$

We assumed a mean-field model that $Q$ factorizes as $$Q(\beta) = \prod_{j=1}^P Q_j(\beta_j).$$ Rather than identify optimal $Q_j$ through CAVI, we will first assume $Q_j := \text{Laplace}(\mu_j, b_j)$. Next, to identify updates for each $\mu_j, b_j$, we take the derivative of the ELBO with respect to each; however the gradient of the ELBO requires knowing $\mu_j, b_j$, which causes challenges.

### 2.1
Re-write the ELBO as a deterministic transformation of $\beta_j$ using location-scale rules (i.e. reparameterization trick)

### 2.2
Implement the above by performing stochastic VI to optimize the ELBO by sampling.



**Reasoning**:
Implement the CAVI algorithm with ELBO tracking and convergence checks as specified in the instructions.

